In [2]:
import pandas as pd 
import numpy as np

In [3]:
sesh_df = pd.read_csv('sessions.csv')
trans_df = pd.read_csv('transactions.csv')

In [25]:
sesh_df.columns

Index(['machine_id', 'site_session_id', 'user_session_id', 'domain_id',
       'ref_domain__name', 'pages_viewed', 'duration', 'event_date',
       'event_time', 'hoh_most_education', 'census_region', 'household_size',
       'hoh_oldest_age', 'household_income', 'children', 'racial_background',
       'connection_speed', 'hispanic', 'zip_code', 'domain_name'],
      dtype='object')

# Data Cleaning: sessions

In [26]:
cleaned_sesh_df = sesh_df.copy()

In [27]:
# convert unknown values into null

# Define a dictionary mapping each column to a list of values to treat as unknown
unknown_values = {
    'racial_background': [99],
    'census_region': [99],
    'household_size': [-88, 99],
    'hoh_oldest_age': [99],
    'household_income': [99],
    'hoh_most_education': [-88, 99]
}

# Replace the specified values with np.nan in the DataFrame
cleaned_sesh_df.replace(unknown_values, np.nan, inplace=True)

from datetime import datetime

# convert event_time to python.time object instead of string
cleaned_sesh_df['event_time'] = cleaned_sesh_df['event_time'].apply(
    lambda x: datetime.strptime(x, "%H:%M:%S").time() if pd.notnull(x) and isinstance(x, str) else None
)

In [28]:
# dropping domain_name as it's in the trans_df... redundant info
cleaned_sesh_df = cleaned_sesh_df[cleaned_sesh_df['domain_name'] == 'ebay.com']
cleaned_sesh_df = cleaned_sesh_df.drop(['domain_name'], axis=1)

# missing percent
cleaned_sesh_df.isnull().mean() * 100

# Note: `hoh_most_education` has around a 60% null percentage, which makes it somewhat unreasonable to impute the missing data.

# attempt to drop the hoh_most_education column
cleaned_sesh_df = cleaned_sesh_df.drop(['hoh_most_education'], axis=1)

In [29]:

# dropping null rows of columns with very small null percentages
# cleaned_sesh_df = cleaned_sesh_df.dropna(subset=['pages_viewed', 'duration', 'event_date', 'event_time', 'hoh_most_education', 'census_region', 'hoh_oldest_age', 'household_income', 'children', 'racial_background', 'hispanic'])

cleaned_sesh_df = cleaned_sesh_df.dropna(subset=['pages_viewed', 'duration', 'event_date', 'event_time', 'census_region', 'hoh_oldest_age', 'household_income', 'children', 'racial_background', 'hispanic'])

# converting ref_domain_name column to a binary flag
cleaned_sesh_df['has_referral'] = cleaned_sesh_df['ref_domain__name'].notnull().astype(int)

cleaned_sesh_df = cleaned_sesh_df.drop(['ref_domain__name'], axis=1)

cleaned_sesh_df = cleaned_sesh_df.reset_index(drop=True)

# imputing household_size, connection_speed, zip_code, hoh_most_education based on census_region and their respective distributions
def impute_by_region_mode(group, col):
  mode_val = group[col].mode()

  if not mode_val.empty:
    group[col].fillna(mode_val[0], inplace=True)

  return group

def impute_by_region_median(group, col):
  median_val = group[col].median()

  group[col].fillna(median_val, inplace=True)

  return group



cleaned_sesh_df = cleaned_sesh_df.groupby('census_region', group_keys=False).apply(lambda g: impute_by_region_mode(g, 'connection_speed')).reset_index(drop=True)

cleaned_sesh_df = cleaned_sesh_df.groupby('census_region', group_keys=False).apply(lambda g: impute_by_region_median(g, 'household_size'))

cleaned_sesh_df = cleaned_sesh_df.groupby('census_region', group_keys=False).apply(lambda g: impute_by_region_mode(g, 'zip_code'))

# cleaned_sesh_df = cleaned_sesh_df.groupby('census_region', group_keys=False).apply(lambda g: impute_by_region_mode(g, 'hoh_most_education'))

## Creating new features

# cleaned_sesh_df

# combine event_date and event_time
cleaned_sesh_df['transaction_datetime'] = pd.to_datetime(
    cleaned_sesh_df['event_date'].astype(str) + ' ' + cleaned_sesh_df['event_time'].astype(str)
)

cleaned_sesh_df = cleaned_sesh_df.drop(['event_date', 'event_time'], axis=1)

# note that we are not creating the indicator features relating to date-time because those are already in trans_df

# calculating the "attention" for each page
cleaned_sesh_df['avg_time_second_per_page'] = (cleaned_sesh_df['duration'] * 60 )/ cleaned_sesh_df['pages_viewed']

cleaned_sesh_df['pages_per_second'] = cleaned_sesh_df['pages_viewed'] / (cleaned_sesh_df['duration'] * 60) # duration is given in minutes

C:\Users\knich\AppData\Local\Temp\ipykernel_15824\3928495181.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  group[col].fillna(mode_val[0], inplace=True)
C:\Users\knich\AppData\Local\Temp\ipykernel_15824\3928495181.py:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after

In [31]:
len(cleaned_sesh_df)

2798922

## Data Cleaning: transactions

In [85]:
cleaned_trans_df = trans_df.copy()

In [86]:
cleaned_trans_df = cleaned_trans_df.drop(['site_session_id', 'prod_name', 'total_transactions_2020', 'total_transactions_2021', 'total_transactions_2022', 'total_transactions_2023'], axis=1)
cleaned_trans_df = cleaned_trans_df.dropna()

In [87]:
cleaned_trans_df = cleaned_trans_df[cleaned_trans_df['domain_name'].isin(['ebay.com'])]

In [88]:
from datetime import datetime

# convert event_time to python.time object instead of string
cleaned_trans_df['event_time'] = cleaned_trans_df['event_time'].apply(
    lambda x: datetime.strptime(x, "%H:%M:%S").time() if pd.notnull(x) and isinstance(x, str) else None
)

## Creating new features

In [89]:
# combine event_date and event_time
cleaned_trans_df['transaction_datetime'] = pd.to_datetime(
    cleaned_trans_df['event_date'].astype(str) + ' ' + cleaned_trans_df['event_time'].astype(str)
)

cleaned_trans_df = cleaned_trans_df.drop(['event_date', 'event_time'], axis=1)

In [90]:
# generating date time related features
cleaned_trans_df['transaction_hour'] = cleaned_trans_df['transaction_datetime'].dt.hour
cleaned_trans_df['transaction_dayofweek'] = cleaned_trans_df['transaction_datetime'].dt.dayofweek  # Monday=0, Sunday=6
cleaned_trans_df['transaction_month'] = cleaned_trans_df['transaction_datetime'].dt.month

cleaned_trans_df['is_weekend'] = cleaned_trans_df['transaction_dayofweek'].isin([5, 6]).astype(int)

In [91]:
cleaned_trans_df['basket_vs_product_diff'] = cleaned_trans_df['basket_tot'] - cleaned_trans_df['prod_totprice']

In [92]:
# r'\.com$' matches '.com' at the end of the string
cleaned_trans_df['domain_name'] = cleaned_trans_df['domain_name'].str.replace(r'\.com$', '', regex=True)

# Create one-hot encoding for the modified column
cleaned_trans_df = pd.get_dummies(cleaned_trans_df, columns=['domain_name'], prefix='domain')

In [18]:
len(cleaned_trans_df)

219118

# Merging dataframes 

In [33]:
cleaned_sesh_df.head()

,machine_id,site_session_id,user_session_id,domain_id,pages_viewed,duration,census_region,household_size,hoh_oldest_age,household_income,children,racial_background,connection_speed,hispanic,zip_code,has_referral,transaction_datetime,avg_time_second_per_page,pages_per_second
0,288084937,6729633401426801739,2412880849370001,1.058424e+19,14.0,14.0,3.0,2.0,11.0,16.0,0.0,1.0,1.0,0.0,28428.0,0,2020-01-01 00:00:02,60.000000,0.016667
1,288912140,1092751701181111355,2412889121400001,1.058424e+19,34.0,31.0,1.0,3.0,11.0,16.0,1.0,1.0,1.0,0.0,13326.0,0,2020-01-01 00:00:02,54.705882,0.018280
2,274888709,5148178385048755113,2412748887090001,1.058424e+19,21.0,21.0,3.0,4.0,11.0,12.0,1.0,5.0,1.0,1.0,33145.0,0,2020-01-01 00:00:04,60.000000,0.016667
3,282355046,6924161971643885156,2412823550460001,1.058424e+19,20.0,11.0,1.0,2.0,10.0,12.0,1.0,1.0,1.0,1.0,4074.0,0,2020-01-01 00:00:09,33.000000,0.030303
4,297138715,3905965572138445730,2412971387150001,1.058424e+19,16.0,14.0,4.0,2.0,10.0,15.0,1.0,1.0,1.0,0.0,85260.0,0,2020-01-01 00:00:09,52.500000,0.019048


In [34]:
cleaned_trans_df.head()

,machine_id,prod_category_id,domain_id,prod_qty,prod_totprice,basket_tot,transaction_datetime,transaction_hour,transaction_dayofweek,transaction_month,is_weekend,basket_vs_product_diff,domain_ebay
20,300908044,3004000000,1.058424e+19,1.0,8.98,9.78,2020-01-01 00:12:58,0,2,1,0,0.80,True
33,300908044,3004010078,1.058424e+19,2.0,12.76,21.51,2020-01-01 00:18:02,0,2,1,0,8.75,True
47,300908044,3004000000,1.058424e+19,1.0,4.67,5.08,2020-01-01 00:28:00,0,2,1,0,0.41,True
51,286745732,5006019207,1.058424e+19,1.0,4.99,4.99,2020-01-01 00:37:00,0,2,1,0,0.00,True
56,290603168,1002003044,1.058424e+19,1.0,15.00,18.78,2020-01-01 00:40:23,0,2,1,0,3.78,True


In [ ]:
cleaned_sesh_df.head()

In [ ]:
# unique_ids = cleaned_sesh_df['machine_id'].unique()
# 
# # Sample 10% of the machine_ids (you can adjust the frac value)
# sampled_ids = pd.Series(unique_ids).sample(frac=0.1, random_state=42)
# 
# # Filter the original DataFrame to keep only transactions from those sampled machine_ids
# sampled_sessions = cleaned_sesh_df[cleaned_sesh_df['machine_id'].isin(sampled_ids)]

In [52]:
cleaned_sesh_df = pd.read_csv('cleaned_sesh_df.csv')

In [53]:
cleaned_sesh_df['event_date'] = pd.to_datetime(cleaned_sesh_df['transaction_datetime']).dt.date
cleaned_trans_df['event_date'] = pd.to_datetime(cleaned_trans_df['transaction_datetime']).dt.date

In [54]:
merged_df = pd.merge(
    cleaned_trans_df,
    cleaned_sesh_df,
    on=['machine_id', 'event_date'],
    how='right'  # or 'left' if you want all transactions with session info
)

In [58]:
inner_merged_df = pd.merge(
    cleaned_trans_df,
    cleaned_sesh_df,
    on=['machine_id', 'event_date'],
    how='inner'  # or 'left' if you want all transactions with session info
)

In [56]:
len(merged_df)

2965242

In [59]:
len(inner_merged_df)

440132

In [44]:
merged_df['is_transacted'] = ~merged_df['transaction_datetime_x'].isna()

In [ ]:
merged_df.head()

In [47]:
merged_df.columns

Index(['machine_id', 'prod_category_id', 'prod_qty', 'prod_totprice',
       'basket_tot', 'transaction_hour', 'transaction_dayofweek',
       'transaction_month', 'is_weekend', 'basket_vs_product_diff',
       'domain_ebay', 'event_date', 'pages_viewed', 'duration',
       'census_region', 'household_size', 'hoh_oldest_age', 'household_income',
       'children', 'racial_background', 'connection_speed', 'hispanic',
       'has_referral', 'avg_time_second_per_page', 'pages_per_second',
       'is_transacted'],
      dtype='object')

In [46]:
columns_to_drop = [
    'domain_id_x', 'domain_id_y',  # redundant IDs
    'transaction_datetime_x', 'transaction_datetime_y',  # keep only session-level info for clustering
    'zip_code',  # high-cardinality, noisy
    'site_session_id', 'user_session_id'  # not useful for modeling
]
merged_df.drop(columns=columns_to_drop, inplace=True)

In [49]:
len(merged_df)

2965242

### Double merge with categories 

In [93]:
categories = pd.read_csv('product_categories.csv')

In [62]:
merged_df.columns

Index(['machine_id', 'prod_category_id', 'domain_id_x', 'prod_qty',
       'prod_totprice', 'basket_tot', 'transaction_datetime_x',
       'transaction_hour', 'transaction_dayofweek', 'transaction_month',
       'is_weekend', 'basket_vs_product_diff', 'domain_ebay', 'event_date',
       'site_session_id', 'user_session_id', 'domain_id_y', 'pages_viewed',
       'duration', 'census_region', 'household_size', 'hoh_oldest_age',
       'household_income', 'children', 'racial_background', 'connection_speed',
       'hispanic', 'zip_code', 'ref_domain_NoReferral', 'ref_domain_Others',
       'ref_domain_bing.com', 'ref_domain_duckduckgo.com',
       'ref_domain_ebay.com', 'ref_domain_facebook.com',
       'ref_domain_google.com', 'ref_domain_googlesyndication.com',
       'ref_domain_msn.com', 'ref_domain_redbrain.shop',
       'ref_domain_yahoo.com', 'ref_domain_youtube.com',
       'transaction_datetime_y', 'avg_time_second_per_page',
       'pages_per_second'],
      dtype='object')

In [94]:
categories.columns

Index(['Product Category ID', 'Report Category', 'Item Category',
       'Item Sub-Category'],
      dtype='object')

In [50]:
merged_df.head()

,machine_id,prod_category_id,prod_qty,prod_totprice,basket_tot,transaction_hour,transaction_dayofweek,transaction_month,is_weekend,basket_vs_product_diff,...,hoh_oldest_age,household_income,children,racial_background,connection_speed,hispanic,has_referral,avg_time_second_per_page,pages_per_second,is_transacted
0,288084937,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,11.0,16.0,0.0,1.0,1.0,0.0,0,60.000000,0.016667,False
1,288912140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,11.0,16.0,1.0,1.0,1.0,0.0,0,54.705882,0.018280,False
2,274888709,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,11.0,12.0,1.0,5.0,1.0,1.0,0,60.000000,0.016667,False
3,282355046,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,10.0,12.0,1.0,1.0,1.0,1.0,0,33.000000,0.030303,False
4,297138715,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,10.0,15.0,1.0,1.0,1.0,0.0,0,52.500000,0.019048,False


In [153]:
double_merged_df = pd.merge(
    merged_df,
    categories,
    left_on= 'prod_category_id',
    right_on= 'Product Category ID',
    how='left'  # or 'left' if you want all transactions with session info
)
double_merged_df.drop(columns = ['Product Category ID', 'Report Category', 
       'Item Sub-Category'])

,machine_id,prod_category_id,domain_id_x,prod_qty,prod_totprice,basket_tot,transaction_datetime_x,transaction_hour,transaction_dayofweek,transaction_month,...,ref_domain_google.com,ref_domain_googlesyndication.com,ref_domain_msn.com,ref_domain_redbrain.shop,ref_domain_yahoo.com,ref_domain_youtube.com,transaction_datetime_y,avg_time_second_per_page,pages_per_second,Item Category
0,288084937,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,...,False,False,False,False,False,False,2020-01-01 00:00:02,60.000000,0.016667,NaN
1,288912140,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,...,False,False,False,False,False,False,2020-01-01 00:00:02,54.705882,0.018280,NaN
2,274888709,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,...,False,False,False,False,False,False,2020-01-01 00:00:04,60.000000,0.016667,NaN
3,282355046,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,...,False,False,False,False,False,False,2020-01-01 00:00:09,33.000000,0.030303,NaN
4,297138715,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,...,False,False,False,False,False,False,2020-01-01 00:00:09,52.500000,0.019048,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2965237,302332442,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,...,False,False,False,False,False,False,1910-04-01 17:00:29,102.000000,0.009804,NaN
2965238,301312440,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,...,False,False,False,False,False,False,2042-12-07 18:59:35,25.000000,0.040000,NaN
2965239,309738893,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,...,False,False,False,False,False,False,1966-01-07 01:56:55,60.000000,0.016667,NaN
2965240,303646530,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,...,False,False,False,False,False,False,2006-10-15 22:05:14,60.000000,0.016667,NaN


In [154]:
double_merged_df.drop(columns = ['transaction_datetime_x', 'transaction_datetime_y', 'is_weekend'], inplace=True)

double_merged_df['event_date'] = pd.to_datetime(double_merged_df['event_date'])
double_merged_df['dayofweek'] = double_merged_df['event_date'].dt.dayofweek 

double_merged_df['is_weekend'] = double_merged_df['transaction_dayofweek'].isin([5, 6]).astype(int)
double_merged_df['is_weekday'] = (double_merged_df['is_weekend'] == 0).astype(int)

double_merged_df.drop(columns = ['dayofweek'], inplace=True)

In [155]:
double_merged_df['Item Category'] = double_merged_df['Item Category'].fillna('NAN')

# One-hot encode the column
item_cat_ohe = pd.get_dummies(double_merged_df['Item Category'], prefix='item_cat')

# Merge back to the main DataFrame
double_merged_df = pd.concat([double_merged_df, item_cat_ohe], axis=1)

In [156]:
double_merged_df.columns

Index(['machine_id', 'prod_category_id', 'domain_id_x', 'prod_qty',
       'prod_totprice', 'basket_tot', 'transaction_hour',
       'transaction_dayofweek', 'transaction_month', 'basket_vs_product_diff',
       'domain_ebay', 'event_date', 'site_session_id', 'user_session_id',
       'domain_id_y', 'pages_viewed', 'duration', 'census_region',
       'household_size', 'hoh_oldest_age', 'household_income', 'children',
       'racial_background', 'connection_speed', 'hispanic', 'zip_code',
       'ref_domain_NoReferral', 'ref_domain_Others', 'ref_domain_bing.com',
       'ref_domain_duckduckgo.com', 'ref_domain_ebay.com',
       'ref_domain_facebook.com', 'ref_domain_google.com',
       'ref_domain_googlesyndication.com', 'ref_domain_msn.com',
       'ref_domain_redbrain.shop', 'ref_domain_yahoo.com',
       'ref_domain_youtube.com', 'avg_time_second_per_page',
       'pages_per_second', 'Product Category ID', 'Report Category',
       'Item Category', 'Item Sub-Category', 'is_weekend', 

In [106]:
# Step 1: Handle NaNs for numeric fields (to avoid issues during aggregation)
# Fill NaNs in transaction columns with 0s, since no transaction occurred in some sessions
double_merged_df['basket_tot'] = double_merged_df['basket_tot'].fillna(0)
double_merged_df['prod_totprice'] = double_merged_df['prod_totprice'].fillna(0)
double_merged_df['prod_qty'] = double_merged_df['prod_qty'].fillna(0)

# Step 2: Create a "successful transaction" indicator
double_merged_df['successful_transaction'] = (double_merged_df['prod_totprice'] > 0).astype(int)


In [123]:
customer_grouped = double_merged_df.groupby('machine_id').agg({
    'basket_tot': ['mean', 'sum'],
    'prod_totprice': ['mean', 'sum'],
    'prod_qty': ['mean', 'sum'],
    'successful_transaction': 'sum',
    'site_session_id': 'count',
    'is_weekday': 'sum',
    'is_weekend': 'sum'
})

# Now flatten column names
customer_grouped.columns = [
    'avg_basket_price', 'sum_basket_price',
    'avg_product_price', 'sum_product_price',
    'avg_product_qty', 'sum_pdt_qty',
    'num_successful_transactions',
    'num_total_sessions',
    'total_weekday_sessions',
    'total_weekend_sessions'
]

# Conversion metrics
customer_grouped['num_unsuccessful_transactions'] = customer_grouped['num_total_sessions'] - customer_grouped['num_successful_transactions']
customer_grouped['conversion_rate'] = customer_grouped['num_successful_transactions'] / customer_grouped['num_total_sessions']

# ========== 3. AGGREGATE ITEM CATEGORY PURCHASES ==========
item_cat_cols = [col for col in double_merged_df.columns if col.startswith('item_cat_')]
item_cat_sums = double_merged_df.groupby('machine_id')[item_cat_cols].sum()

# Add to main table
customer_grouped = pd.concat([customer_grouped, item_cat_sums], axis=1)

# ========== 4. DEMOGRAPHIC FEATURES ==========
demo_cols = [
    'household_size', 'hoh_oldest_age', 'household_income', 'children',
    'racial_background', 'connection_speed', 'hispanic', 'census_region', 'zip_code'
]

# Define custom mode aggregator
def most_frequent(series):
    try:
        return series.mode().iloc[0]
    except:
        return np.nan

# Group by machine_id and take the mode (most common value)
demographics_agg = double_merged_df.groupby('machine_id')[demo_cols].agg(most_frequent)

# Merge with customer_grouped
customer_grouped = customer_grouped.merge(demographics_agg, on='machine_id', how='left')
customer_grouped.rename(columns = {'num_unsuccessful_transactions': 'num_unsuccessful_sessions'}, inplace=True)

item_cat_cols = [col for col in customer_grouped.columns if col.startswith('item_cat_')]
customer_grouped['num_unique_cat'] = (customer_grouped[item_cat_cols] > 0).sum(axis=1)

customer_grouped = customer_grouped.reset_index()

In [126]:
customer_grouped.drop(columns=['item_cat_NAN'], inplace=True)
customer_grouped['num_unique_cat'] = customer_grouped['num_unique_cat'] - 1
customer_grouped.rename(columns = {'num_unique_cat': 'num_unique_categories'}, inplace=True)

In [177]:
customer_grouped.columns

Index(['machine_id', 'avg_basket_price', 'sum_basket_price',
       'avg_product_price', 'sum_product_price', 'avg_product_qty',
       'sum_pdt_qty', 'num_successful_transactions', 'num_total_sessions',
       'total_weekday_sessions', 'total_weekend_sessions',
       'num_unsuccessful_sessions', 'conversion_rate', 'item_cat_APPAREL',
       'item_cat_APPLIANCES', 'item_cat_ARTS & CRAFTS',
       'item_cat_AUTO & TIRES', 'item_cat_BABY', 'item_cat_BAGS & ACCESSORIES',
       'item_cat_BEAUTY', 'item_cat_BEDDING, BATH, & DECOR', 'item_cat_BOOKS',
       'item_cat_CAMERAS & CAMCORDERS', 'item_cat_COMPUTERS & PRINTERS',
       'item_cat_ELECTRONICS ACCESSORIES', 'item_cat_FLOWERS & GIFTS',
       'item_cat_FOOD & BEVERAGE', 'item_cat_FURNITURE & STORAGE',
       'item_cat_HEALTH', 'item_cat_HOBBIES', 'item_cat_HOME IMPROVEMENT',
       'item_cat_HOME THEATER',
       'item_cat_HOUSEHOLD ESSENTIALS & CLEANING SUPPLIES', 'item_cat_JEWELRY',
       'item_cat_KITCHEN & DINING', 'item_cat_LAW

In [63]:
session_level = merged_df.groupby('site_session_id').agg({
    'pages_viewed': 'max',
    'duration': 'max',
    'basket_tot': 'sum',  # sum of purchases in that session
    'prod_totprice': 'sum',
    'prod_qty': 'sum',
    'prod_category_id': pd.Series.nunique,  # how many distinct categories bought
    'transaction_hour': 'first',
    'transaction_dayofweek': 'first',
    'is_weekend': 'first',
    'avg_time_second_per_page': 'mean',
    'pages_per_second': 'mean',
    'household_income': 'first',
    'hoh_oldest_age': 'first',
    'household_size': 'first',
    'zip_code': 'first',
    'machine_id': 'first'
}).reset_index()

In [64]:
transaction_level = merged_df.groupby(['machine_id', 'transaction_datetime_x']).agg({
    'basket_tot': 'sum',
    'prod_totprice': 'sum',
    'prod_qty': 'sum',
    'prod_category_id': pd.Series.nunique,
    'pages_viewed': 'max',
    'duration': 'max',
    'transaction_hour': 'first',
    'transaction_dayofweek': 'first',
    'is_weekend': 'first',
    'site_session_id': 'first'
}).reset_index()

In [83]:
customer_level = merged_df.groupby('machine_id').agg({
    'basket_tot': 'sum',
    'prod_totprice': 'sum',
    'prod_qty': 'sum',
    'site_session_id': pd.Series.nunique,
    'transaction_datetime_x': pd.Series.nunique,
    'pages_viewed': 'mean',
    'duration': 'mean',
    'prod_category_id': pd.Series.nunique,

    # Demographic features — retain as-is
    'household_size': 'first',
    'hoh_oldest_age': 'first',
    'household_income': 'first',
    'children': 'first',
    'racial_background': 'first',
    'connection_speed': 'first',
    'hispanic': 'first',
    'zip_code': 'first'
}).reset_index()


In [84]:
customer_level.columns

Index(['machine_id', 'basket_tot', 'prod_totprice', 'prod_qty',
       'site_session_id', 'transaction_datetime_x', 'pages_viewed', 'duration',
       'prod_category_id', 'household_size', 'hoh_oldest_age',
       'household_income', 'children', 'racial_background', 'connection_speed',
       'hispanic'],
      dtype='object')

In [ ]:
customer_level.rename(columns)

In [66]:
inner_session_level = inner_merged_df.groupby('site_session_id').agg({
    'pages_viewed': 'max',
    'duration': 'max',
    'basket_tot': 'sum',  # sum of purchases in that session
    'prod_totprice': 'sum',
    'prod_qty': 'sum',
    'prod_category_id': pd.Series.nunique,  # how many distinct categories bought
    'transaction_hour': 'first',
    'transaction_dayofweek': 'first',
    'is_weekend': 'first',
    'avg_time_second_per_page': 'mean',
    'pages_per_second': 'mean',
    'household_income': 'first',
    'hoh_oldest_age': 'first',
    'household_size': 'first',
    'zip_code': 'first',
    'machine_id': 'first'
}).reset_index()

In [67]:
inner_transaction_level = inner_merged_df.groupby(['machine_id', 'transaction_datetime_x']).agg({
    'basket_tot': 'sum',
    'prod_totprice': 'sum',
    'prod_qty': 'sum',
    'prod_category_id': pd.Series.nunique,
    'pages_viewed': 'max',
    'duration': 'max',
    'transaction_hour': 'first',
    'transaction_dayofweek': 'first',
    'is_weekend': 'first',
    'site_session_id': 'first'
}).reset_index()

In [71]:
inner_customer_level = inner_merged_df.groupby('machine_id').agg({
    'basket_tot': 'sum',
    'prod_totprice': 'sum',
    'prod_qty': 'sum',
    'site_session_id': pd.Series.nunique,
    'transaction_datetime_x': pd.Series.nunique,  # number of distinct transactions
    'pages_viewed': 'mean',
    'duration': 'mean',
    'prod_category_id': pd.Series.nunique,
    'household_income': 'first',
    'census_region': 'first',
    'connection_speed': 'first'
}).reset_index()

In [68]:
inner_session_level.head()

,site_session_id,pages_viewed,duration,basket_tot,prod_totprice,prod_qty,prod_category_id,transaction_hour,transaction_dayofweek,is_weekend,avg_time_second_per_page,pages_per_second,household_income,hoh_oldest_age,household_size,zip_code,machine_id
0,54178183042785,16.0,52.0,95.43,89.00,1.0,1,1,2,0,195.000000,0.005128,12.0,5.0,2.0,65018.0,357640082
1,81973314778742,65.0,64.0,27.84,22.49,1.0,1,4,6,1,59.076923,0.016927,14.0,7.0,4.0,11222.0,281428058
2,103263156239606,1.0,1.0,43.20,40.00,1.0,1,21,2,0,60.000000,0.016667,13.0,11.0,2.0,29566.0,330392722
3,126281687473810,16.0,7.0,8.89,8.45,1.0,1,22,5,1,26.250000,0.038095,13.0,11.0,2.0,73089.0,354027186
4,173328505776681,2.0,1.0,30.04,27.98,1.0,1,22,1,0,30.000000,0.033333,17.0,9.0,2.0,55917.0,325771980


In [69]:
inner_transaction_level.head()

,machine_id,transaction_datetime_x,basket_tot,prod_totprice,prod_qty,prod_category_id,pages_viewed,duration,transaction_hour,transaction_dayofweek,is_weekend,site_session_id
0,100026976,2022-10-17 03:09:26,129.63,109.71,3.0,1,22.0,29.0,3,0,0,7757747425267377381
1,104679635,2022-05-19 08:54:45,44.04,29.95,1.0,1,3.0,1.0,8,3,0,2961291707642428989
2,123005549,2023-03-14 15:47:05,15.55,14.58,1.0,1,15.0,53.0,15,1,0,1391366339713378262
3,123005549,2023-03-23 15:44:40,341.10,319.90,2.0,1,16.0,69.0,15,3,0,3556845585233678636
4,123005549,2023-04-18 16:40:26,42.64,39.99,1.0,1,9.0,16.0,16,1,0,2520119805771979145


In [77]:
customer_level.rename(columns= {'transaction_datetime_x': 'num_transactions', 'prod_category_id': 'num_prod_category'}, inplace=True)
customer_level.head()

,machine_id,basket_tot,prod_totprice,prod_qty,site_session_id,num_transactions,pages_viewed,duration,num_prod_category,household_income,census_region,connection_speed
0,54388726,0.00,0.00,0.0,6,0,1.666667,1.000000,0,16.0,3.0,0.0
1,76893652,0.00,0.00,0.0,53,0,3.740741,3.981481,0,13.0,1.0,1.0
2,84499675,0.00,0.00,0.0,3,0,3.000000,1.000000,0,16.0,1.0,0.0
3,98480988,0.00,0.00,0.0,186,0,10.304813,7.390374,0,14.0,1.0,1.0
4,100026976,129.63,109.71,3.0,1079,1,15.065257,20.556066,1,15.0,3.0,1.0


In [79]:
len(customer_level)

101426

In [141]:
double_merged_df.columns

Index(['machine_id', 'prod_category_id', 'domain_id_x', 'prod_qty',
       'prod_totprice', 'basket_tot', 'transaction_hour',
       'transaction_dayofweek', 'transaction_month', 'basket_vs_product_diff',
       'domain_ebay', 'event_date', 'site_session_id', 'user_session_id',
       'domain_id_y', 'pages_viewed', 'duration', 'census_region',
       'household_size', 'hoh_oldest_age', 'household_income', 'children',
       'racial_background', 'connection_speed', 'hispanic', 'zip_code',
       'ref_domain_NoReferral', 'ref_domain_Others', 'ref_domain_bing.com',
       'ref_domain_duckduckgo.com', 'ref_domain_ebay.com',
       'ref_domain_facebook.com', 'ref_domain_google.com',
       'ref_domain_googlesyndication.com', 'ref_domain_msn.com',
       'ref_domain_redbrain.shop', 'ref_domain_yahoo.com',
       'ref_domain_youtube.com', 'avg_time_second_per_page',
       'pages_per_second', 'Product Category ID', 'Report Category',
       'Item Category', 'Item Sub-Category', 'is_weekend', 

In [178]:
# Copy original
double_merged_df['is_transacted'] = double_merged_df['transaction_hour'].notna().astype(int)
session_df = double_merged_df.copy()

# Identify item category and demographic columns
item_cat_cols = [col for col in session_df.columns if col.startswith('item_cat_')]
demo_cols = [
    'household_size', 'hoh_oldest_age', 'household_income', 'children',
    'racial_background', 'connection_speed', 'hispanic', 'census_region'
]

# Define aggregation
agg_all = {
    'machine_id': 'first',
    'pages_viewed': 'mean',
    'duration': 'mean',
    'avg_time_second_per_page': 'mean',
    'pages_per_second': 'mean',
    'is_transacted': 'max',
    **{col: 'first' for col in demo_cols},  # assumes demographics consistent per session
    **{col: 'sum' for col in item_cat_cols},
}

# Group
session_grouped_all = session_df.groupby('site_session_id').agg(agg_all).reset_index()


In [180]:
session_grouped_all.columns

Index(['site_session_id', 'machine_id', 'pages_viewed', 'duration',
       'avg_time_second_per_page', 'pages_per_second', 'is_transacted',
       'household_size', 'hoh_oldest_age', 'household_income', 'children',
       'racial_background', 'connection_speed', 'hispanic', 'census_region',
       'item_cat_APPAREL', 'item_cat_APPLIANCES', 'item_cat_ARTS & CRAFTS',
       'item_cat_AUTO & TIRES', 'item_cat_BABY', 'item_cat_BAGS & ACCESSORIES',
       'item_cat_BEAUTY', 'item_cat_BEDDING, BATH, & DECOR', 'item_cat_BOOKS',
       'item_cat_CAMERAS & CAMCORDERS', 'item_cat_COMPUTERS & PRINTERS',
       'item_cat_ELECTRONICS ACCESSORIES', 'item_cat_FLOWERS & GIFTS',
       'item_cat_FOOD & BEVERAGE', 'item_cat_FURNITURE & STORAGE',
       'item_cat_HEALTH', 'item_cat_HOBBIES', 'item_cat_HOME IMPROVEMENT',
       'item_cat_HOME THEATER',
       'item_cat_HOUSEHOLD ESSENTIALS & CLEANING SUPPLIES', 'item_cat_JEWELRY',
       'item_cat_KITCHEN & DINING', 'item_cat_LAWN CARE & EQUIPMENT',
     

In [179]:
# Copy original
double_merged_df['is_transacted'] = double_merged_df['transaction_hour'].notna().astype(int)
session_df = double_merged_df.copy()

# Identify demographics
demo_cols = [
    'household_size', 'hoh_oldest_age', 'household_income', 'children',
    'racial_background', 'connection_speed', 'hispanic', 'census_region'
]

# Identify and filter item_cat columns
item_cat_cols = [col for col in session_df.columns if col.startswith('item_cat_')]
item_cat_cols = [col for col in item_cat_cols if 'NAN' not in col.upper()]

# Top 5 defined manually or use this:
# top5 = session_df[item_cat_cols].sum().sort_values(ascending=False).head(5).index.tolist()
top5 = ['item_cat_BEAUTY', 'item_cat_HOME IMPROVEMENT', 'item_cat_AUTO & TIRES',
        'item_cat_HOBBIES', 'item_cat_TOYS']
others = [col for col in item_cat_cols if col not in top5]

# Create 'item_cat_OTHERS'
session_df['item_cat_OTHERS'] = session_df[others].sum(axis=1)
session_df.drop(columns=others, inplace=True)

# Now define aggregation (after filtering)
agg_all = {
    'machine_id': 'first',
    'pages_viewed': 'mean',
    'duration': 'mean',
    'avg_time_second_per_page': 'mean',
    'pages_per_second': 'mean',
    'is_transacted': 'max',
    **{col: 'first' for col in demo_cols},
    **{col: 'sum' for col in top5 + ['item_cat_OTHERS']},
}

# Group by session
session_grouped_top5 = session_df.groupby('site_session_id').agg(agg_all).reset_index()


In [181]:
session_grouped_top5.head()

,site_session_id,machine_id,pages_viewed,duration,avg_time_second_per_page,pages_per_second,is_transacted,household_size,hoh_oldest_age,household_income,...,racial_background,connection_speed,hispanic,census_region,item_cat_BEAUTY,item_cat_HOME IMPROVEMENT,item_cat_AUTO & TIRES,item_cat_HOBBIES,item_cat_TOYS,item_cat_OTHERS
0,1564489242478,331798906,1.0,1.0,60.000000,0.016667,0,4.0,6.0,13.0,...,5.0,1.0,0.0,1.0,0,0,0,0,0,0
1,7130168806244,332645392,4.0,4.0,60.000000,0.016667,0,2.0,5.0,13.0,...,1.0,1.0,0.0,3.0,0,0,0,0,0,0
2,9605097083885,281082467,9.0,21.0,140.000000,0.007143,0,2.0,10.0,16.0,...,1.0,1.0,0.0,3.0,0,0,0,0,0,0
3,10379216085485,301323313,11.0,5.0,27.272727,0.036667,0,1.0,8.0,15.0,...,1.0,1.0,0.0,3.0,0,0,0,0,0,0
4,15887259144645,317759847,15.0,15.0,60.000000,0.016667,0,2.0,11.0,12.0,...,1.0,1.0,0.0,2.0,0,0,0,0,0,0


In [190]:
# Copy original
double_merged_df['is_transacted'] = double_merged_df['transaction_hour'].notna().astype(int)
session_df = double_merged_df.copy()

# Identify demographics
demo_cols = [
    'household_size', 'hoh_oldest_age', 'household_income', 'children',
    'racial_background', 'connection_speed', 'hispanic', 'census_region'
]

# Identify and filter item_cat columns
item_cat_cols = [col for col in session_df.columns if col.startswith('item_cat_')]
item_cat_cols = [col for col in item_cat_cols if 'NAN' not in col.upper()]

# Top 5 defined manually or use this:
total_by_cat = customer_grouped[item_cat_cols].sum().sort_values(ascending=False)
top10 = total_by_cat.head(11).index.tolist()
others = [col for col in item_cat_cols if col not in top10]

# Create 'item_cat_OTHERS'
session_df['item_cat_OTHERS'] = session_df[others].sum(axis=1)
session_df.drop(columns=others, inplace=True)

# Now define aggregation (after filtering)
agg_all = {
    'machine_id': 'first',
    'pages_viewed': 'mean',
    'duration': 'mean',
    'avg_time_second_per_page': 'mean',
    'pages_per_second': 'mean',
    'is_transacted': 'max',
    **{col: 'first' for col in demo_cols},
    **{col: 'sum' for col in top5 + ['item_cat_OTHERS']},
}

# Group by session
session_grouped_top10 = session_df.groupby('site_session_id').agg(agg_all).reset_index()

In [192]:
session_grouped_top10.head()

,site_session_id,machine_id,pages_viewed,duration,avg_time_second_per_page,pages_per_second,is_transacted,household_size,hoh_oldest_age,household_income,...,item_cat_OTHERS,item_cat_BEAUTY,item_cat_HOME IMPROVEMENT,item_cat_AUTO & TIRES,item_cat_HOBBIES,item_cat_TOYS,item_cat_APPAREL,item_cat_ELECTRONICS ACCESSORIES,item_cat_JEWELRY,item_cat_BOOKS
0,1564489242478,331798906,1.0,1.0,60.000000,0.016667,0,4.0,6.0,13.0,...,0,0,0,0,0,0,0,0,0,0
1,7130168806244,332645392,4.0,4.0,60.000000,0.016667,0,2.0,5.0,13.0,...,0,0,0,0,0,0,0,0,0,0
2,9605097083885,281082467,9.0,21.0,140.000000,0.007143,0,2.0,10.0,16.0,...,0,0,0,0,0,0,0,0,0,0
3,10379216085485,301323313,11.0,5.0,27.272727,0.036667,0,1.0,8.0,15.0,...,0,0,0,0,0,0,0,0,0,0
4,15887259144645,317759847,15.0,15.0,60.000000,0.016667,0,2.0,11.0,12.0,...,0,0,0,0,0,0,0,0,0,0


## Exporting data to csv

In [80]:
inner_session_level.to_csv('session_grouped.csv')
inner_transaction_level.to_csv('transactions_grouped.csv')
customer_level.to_csv('customer_grouped.csv')

In [19]:
cleaned_trans_df.to_csv('cleaned_transactions.csv', index=False)

In [32]:
cleaned_sesh_df.to_csv('cleaned_sessions.csv', index=False)

In [48]:
unique_ids = merged_df['machine_id'].unique()

# Sample 10% of the machine_ids (you can adjust the frac value)
sampled_ids = pd.Series(unique_ids).sample(frac=0.1, random_state=42)

# Filter the original DataFrame to keep only transactions from those sampled machine_ids
sampled_merged = merged_df[merged_df['machine_id'].isin(sampled_ids)]

sampled_merged.to_csv('sampled_merged_df.csv', index=False)

In [ ]:
sample_sesh_df = cleaned_sesh_df.head(40000)

In [ ]:
sample_trans_df = cleaned_trans_df.head(400000)

In [ ]:
sample_sesh_df.to_csv('ss_sessions.csv', index=False)
sample_trans_df.to_csv('ss_transactions.csv', index=False)

In [60]:
inner_merged_df.to_csv('inner_merged_df.csv', index=False)
merged_df.to_csv('left_merged_df.csv', index=False)

In [129]:
customer_grouped.to_csv('final_merged_df.csv', index=False)

In [131]:
customer_grouped.columns

Index(['machine_id', 'avg_basket_price', 'sum_basket_price',
       'avg_product_price', 'sum_product_price', 'avg_product_qty',
       'sum_pdt_qty', 'num_successful_transactions', 'num_total_sessions',
       'total_weekday_sessions', 'total_weekend_sessions',
       'num_unsuccessful_sessions', 'conversion_rate', 'item_cat_APPAREL',
       'item_cat_APPLIANCES', 'item_cat_ARTS & CRAFTS',
       'item_cat_AUTO & TIRES', 'item_cat_BABY', 'item_cat_BAGS & ACCESSORIES',
       'item_cat_BEAUTY', 'item_cat_BEDDING, BATH, & DECOR', 'item_cat_BOOKS',
       'item_cat_CAMERAS & CAMCORDERS', 'item_cat_COMPUTERS & PRINTERS',
       'item_cat_ELECTRONICS ACCESSORIES', 'item_cat_FLOWERS & GIFTS',
       'item_cat_FOOD & BEVERAGE', 'item_cat_FURNITURE & STORAGE',
       'item_cat_HEALTH', 'item_cat_HOBBIES', 'item_cat_HOME IMPROVEMENT',
       'item_cat_HOME THEATER',
       'item_cat_HOUSEHOLD ESSENTIALS & CLEANING SUPPLIES', 'item_cat_JEWELRY',
       'item_cat_KITCHEN & DINING', 'item_cat_LAW

In [176]:
len(session_grouped_top5)

2771371

In [182]:
session_grouped_all.to_csv('session_grouped_all.csv', index=False)
session_grouped_top5.to_csv('session_grouped_top5.csv', index=False)

In [193]:
session_grouped_top10.to_csv('session_grouped_top10.csv', index=False)

# EDA 

In [132]:
item_cat_cols = [col for col in customer_grouped.columns if col.startswith('item_cat_')]

# Step 1: Sum each category column to get total counts
category_totals = customer_grouped[item_cat_cols].sum().sort_values(ascending=False)

# Step 2: Get top 5 category names
top_5 = category_totals.head(5).index.tolist()

# Step 3: Create a new column 'item_cat_OTHERS' as the sum of non-top-5 categories
others = [col for col in item_cat_cols if col not in top_5]
customer_grouped['item_cat_OTHERS'] = customer_grouped[others].sum(axis=1)

# Step 4: Keep only top 5 + OTHERS
final_cols = top_5 + ['item_cat_OTHERS']
df_top5 = customer_grouped[final_cols]

In [170]:
df_top5.columns


Index(['item_cat_BEAUTY', 'item_cat_HOME IMPROVEMENT', 'item_cat_AUTO & TIRES',
       'item_cat_HOBBIES', 'item_cat_TOYS', 'item_cat_OTHERS'],
      dtype='object')

In [138]:
top5_cats_df = customer_grouped.copy()

# Step 2: Identify all item category columns
item_cat_cols = [col for col in top5_cats_df.columns if col.startswith('item_cat_')]

# Step 3: Get top 5 based on total sum
total_by_cat = top5_cats_df[item_cat_cols].sum().sort_values(ascending=False)
top5 = total_by_cat.head(5).index.tolist()
others = [col for col in item_cat_cols if col not in top5]

# Step 4: Add 'item_cat_OTHERS'
top5_cats_df['item_cat_OTHERS'] = top5_cats_df[others].sum(axis=1)

# Step 5: Drop all category columns not in top 5
drop_cols = [col for col in item_cat_cols if col not in top5]
top5_cats_df.drop(columns=drop_cols, inplace=True)

In [139]:
top5_cats_df.head()

,machine_id,avg_basket_price,sum_basket_price,avg_product_price,sum_product_price,avg_product_qty,sum_pdt_qty,num_successful_transactions,num_total_sessions,total_weekday_sessions,...,hoh_oldest_age,household_income,children,racial_background,connection_speed,hispanic,census_region,zip_code,num_unique_categories,item_cat_OTHERS
0,54388726,0.000000,0.00,0.000000,0.00,0.000000,0.0,0,6,6,...,5.0,16.0,0.0,1.0,0.0,1.0,3.0,33130.0,0,0
1,76893652,0.000000,0.00,0.000000,0.00,0.000000,0.0,0,54,54,...,7.0,13.0,1.0,1.0,1.0,0.0,1.0,6058.0,0,0
2,84499675,0.000000,0.00,0.000000,0.00,0.000000,0.0,0,3,3,...,11.0,16.0,1.0,5.0,0.0,0.0,1.0,7601.0,0,0
3,98480988,0.000000,0.00,0.000000,0.00,0.000000,0.0,0,187,187,...,11.0,14.0,1.0,1.0,1.0,0.0,1.0,2871.0,0,0
4,100026976,0.119145,129.63,0.100836,109.71,0.002757,3.0,3,1088,1088,...,11.0,15.0,0.0,1.0,1.0,0.0,3.0,31405.0,1,0


In [186]:
top5_cats_df.to_csv('customer_top5.csv', index=False)

In [184]:
top10_cats_df = customer_grouped.copy()

# Step 2: Identify all item category columns
item_cat_cols = [col for col in top10_cats_df.columns if col.startswith('item_cat_')]

# Step 3: Get top 5 based on total sum
total_by_cat = top10_cats_df[item_cat_cols].sum().sort_values(ascending=False)
top5 = total_by_cat.head(10).index.tolist()
others = [col for col in item_cat_cols if col not in top5]

# Step 4: Add 'item_cat_OTHERS'
top10_cats_df['item_cat_OTHERS'] = top10_cats_df[others].sum(axis=1)

# Step 5: Drop all category columns not in top 5
drop_cols = [col for col in item_cat_cols if col not in top5]
top10_cats_df.drop(columns=drop_cols, inplace=True)

In [187]:
top10_cats_df.to_csv('customer_top10.csv', index=False)